# Mercor Cheating Detection — Cost-Aware Classification Pipeline

**Team:** Saurav Rijal, Shivendra Bhagat

### Problem
Binary classification to detect cheating during online AI-driven interviews.  
Objective: **minimize total economic cost**.

| Decision | Legit user | Cheater |
|---|---|---|
| Auto-pass `prob < t_pass` | \$0 | \$600 FN |
| Manual review | \$150 FP | \$5 TP |
| Auto-block `prob ≥ t_block` | \$300 FP | \$0 |

### Approach
1. **Preprocessing** — median imputation, missingness flags, log transforms, z-score outlier flags, MI-guided interaction terms
2. **Graph features** — degree centrality, 1-hop/2-hop cheating density, ghost-user ratio, Louvain community cheat rate
3. **Model progression** — LogReg → Decision Tree → Random Forest → XGBoost → LightGBM → Ensemble blend
4. **Calibration** — per-fold Temperature / Platt / Isotonic (auto-selected by positive count)
5. **Threshold sweep** — grid-search over (t_pass, t_block) to minimize cost on OOF predictions
6. **Semi-supervised** — pseudo-labeling of high-confidence clean unlabeled users

## Setup

In [ ]:
!pip install xgboost lightgbm python-louvain -q

In [ ]:
import os, copy, time, glob, warnings
from pathlib import Path
from itertools import combinations
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
from scipy.optimize import minimize_scalar
from scipy.special import expit, logit
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')

# ── Feature groups ────────────────────────────────────────────────────────────
ALL_RAW         = [f'feature_{i:03d}' for i in range(1, 19)]
BINARY_FEATURES = ['feature_007', 'feature_011', 'feature_013', 'feature_014']
NUMERIC_RAW     = [f for f in ALL_RAW if f not in BINARY_FEATURES]
HIGH_MISSING    = [
    'feature_001', 'feature_002', 'feature_003',
    'feature_007', 'feature_008', 'feature_009',
    'feature_010', 'feature_011', 'feature_012',
    'feature_013', 'feature_014', 'feature_017', 'feature_018',
]
LOG_FEATURES    = ['feature_010', 'feature_016', 'feature_015']

# ── Cost constants ────────────────────────────────────────────────────────────
COST_FN        = 600
COST_FP_BLOCK  = 300
COST_FP_REVIEW = 150
COST_TP_REVIEW = 5

# ── Training constants ────────────────────────────────────────────────────────
COST_RATIO   = 2.0
N_SPLITS     = 5
RANDOM_STATE = 42

# ── Paths ─────────────────────────────────────────────────────────────────────
data_dirs = glob.glob('/kaggle/input/**/train.csv', recursive=True)
DATA_DIR  = Path(os.path.dirname(data_dirs[0])) if data_dirs else Path('data')
os.makedirs('/kaggle/working', exist_ok=True)
print(f'Data dir: {DATA_DIR}')

## 1 — Data Loading

In [ ]:
print('[data] Loading ...')
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')

n_before = len(train)
train = train.drop_duplicates(subset=['user_hash'])
if len(train) < n_before:
    print(f'[data] Removed {n_before - len(train)} duplicate rows')

labeled_mask    = train['is_cheating'].notna()
train_labeled   = train[labeled_mask].copy()
train_unlabeled = train[~labeled_mask].copy()

print(
    f'Train: {len(train):,}  |  Labeled: {len(train_labeled):,}  '
    f'|  Unlabeled: {len(train_unlabeled):,}  |  Test: {len(test):,}'
)
print(f'Cheating rate (labeled): {train_labeled["is_cheating"].mean():.4f}')

## 2 — Feature Engineering

In [ ]:
def impute_and_flag(df):
    df = df.copy()
    for col in HIGH_MISSING:
        if col not in df.columns:
            continue
        df[f'{col}_was_missing'] = df[col].isna().astype(np.int8)
        df[col] = df[col].fillna(df[col].median())
    return df


def log_transform(df):
    df = df.copy()
    for col in LOG_FEATURES:
        if col not in df.columns:
            continue
        shift = abs(df[col].min()) + 1.0 if df[col].min() <= 0 else 0.0
        df[f'{col}_log'] = np.log1p(df[col] + shift)
    return df


def add_zscore_flags(df, threshold=3.5):
    df = df.copy()
    for col in NUMERIC_RAW:
        if col not in df.columns:
            continue
        mu, sigma = df[col].mean(), df[col].std()
        if sigma < 1e-9:
            continue
        df[f'{col}_is_extreme'] = ((df[col] - mu).abs() / sigma > threshold).astype(np.int8)
    return df


def add_ratio_features(df):
    df  = df.copy()
    eps = 1e-6
    df['f003_f001_ratio'] = df['feature_003'] / (df['feature_001'] + eps)
    df['f002_f005_ratio'] = df['feature_002'] / (df['feature_005'] + eps)
    df['f008_f009_ratio'] = df['feature_008'] / (df['feature_009'] + eps)
    df['f004_per_f015']   = df['feature_004'] / (df['feature_015'].abs() + eps)

    binary_cols = [c for c in BINARY_FEATURES if c in df.columns]
    df['binary_flag_sum'] = df[binary_cols].sum(axis=1)

    if 'feature_015' in df.columns:
        df['feature_015_gt80'] = (df['feature_015'] > 80).astype(np.int8)
        df['feature_015_bin']  = pd.cut(
            df['feature_015'],
            bins=[-np.inf, 0, 20, 80, 200, np.inf],
            labels=[0, 1, 2, 3, 4],
        ).astype(float)

    if 'feature_013' in df.columns and 'feature_015' in df.columns:
        at_risk = 1 - df['feature_013'].fillna(0)
        df['f013inv_x_f015']      = at_risk * df['feature_015']
        df['f013inv_x_f015_gt80'] = at_risk * df.get('feature_015_gt80', 0)

    if 'feature_013' in df.columns and 'feature_004' in df.columns:
        df['f013inv_x_f004'] = (1 - df['feature_013'].fillna(0)) * df['feature_004']

    return df


def select_top_pairs(df, y, top_k=5):
    cols   = [c for c in NUMERIC_RAW if c in df.columns]
    scores = {}
    for a, b in combinations(cols, 2):
        mi = mutual_info_classif(
            (df[a] * df[b]).values.reshape(-1, 1), y, discrete_features=False
        )[0]
        scores[(a, b)] = mi
    return sorted(scores, key=lambda k: scores[k], reverse=True)[:top_k]


def add_interaction_terms(df, pairs=None):
    DEFAULT_PAIRS = [
        ('feature_001', 'feature_003'), ('feature_004', 'feature_005'),
        ('feature_002', 'feature_006'), ('feature_008', 'feature_009'),
        ('feature_015', 'feature_016'),
    ]
    df = df.copy()
    for a, b in (pairs or DEFAULT_PAIRS):
        if a in df.columns and b in df.columns:
            df[f'{a}_x_{b}'] = df[a] * df[b]
    return df


def build_behavioral_features(df, interaction_pairs=None):
    df = impute_and_flag(df)
    df = log_transform(df)
    df = add_zscore_flags(df)
    df = add_ratio_features(df)
    df = add_interaction_terms(df, interaction_pairs)
    return df


print('[feat] Selecting MI interaction pairs ...')
y_lab     = train_labeled['is_cheating'].astype(int)
X_lab_raw = train_labeled[[c for c in ALL_RAW if c in train_labeled.columns]]
interaction_pairs = select_top_pairs(X_lab_raw.fillna(0), y_lab, top_k=5)
print(f'Interaction pairs: {interaction_pairs}')

train_lab_feat   = build_behavioral_features(train_labeled,   interaction_pairs)
train_unlab_feat = build_behavioral_features(train_unlabeled, interaction_pairs)
test_feat        = build_behavioral_features(test,            interaction_pairs)
print('[feat] Done')

## 3 — Social Graph Features

In [ ]:
graph_path = DATA_DIR / 'social_graph.csv'

if graph_path.exists():
    t0 = time.time()
    print(f'[graph] Reading {graph_path} ...')
    edges     = pd.read_csv(graph_path)
    G         = nx.from_pandas_edgelist(edges, source='user_a', target='user_b')
    node_list = np.array(list(G.nodes()))
    node2idx  = {n: i for i, n in enumerate(node_list)}
    print(f'[graph] nodes={G.number_of_nodes():,}  edges={G.number_of_edges():,}  ({time.time()-t0:.1f}s)')

    # Sparse adjacency
    rows, cols_idx = [], []
    for u, v in G.edges():
        i, j = node2idx[u], node2idx[v]
        rows += [i, j]; cols_idx += [j, i]
    A = sp.csr_matrix(
        (np.ones(len(rows), dtype=np.float32), (rows, cols_idx)),
        shape=(len(node2idx), len(node2idx)),
    )

    cheat_set   = set(train_labeled.loc[train_labeled['is_cheating'] == 1, 'user_hash'])
    labeled_set = set(train_labeled['user_hash'])
    known_users = labeled_set | set(train_unlabeled['user_hash']) | set(test['user_hash'])
    all_users   = (list(train_labeled['user_hash']) +
                   list(train_unlabeled['user_hash']) +
                   list(test['user_hash']))

    n = len(node_list)
    cheat_vec   = np.array([1.0 if nd in cheat_set   else 0.0 for nd in node_list], dtype=np.float32)
    labeled_vec = np.array([1.0 if nd in labeled_set else 0.0 for nd in node_list], dtype=np.float32)
    ghost_vec   = np.array([0.0 if nd in known_users else 1.0 for nd in node_list], dtype=np.float32)

    cheat_1hop   = np.array(A @ cheat_vec).ravel()
    labeled_1hop = np.array(A @ labeled_vec).ravel()
    A2_cheat     = np.array(A @ (A @ cheat_vec)).ravel()
    A2_labeled   = np.array(A @ (A @ labeled_vec)).ravel()
    ghost_1hop   = np.array(A @ ghost_vec).ravel()
    degree_arr   = np.array(A.sum(axis=1)).ravel()
    deg_dict     = dict(G.degree())

    print('[graph] Running community detection ...', flush=True)
    try:
        import community as community_louvain
        partition = community_louvain.best_partition(G, resolution=1.0)
    except ImportError:
        partition = {}
        for i, comp in enumerate(nx.connected_components(G)):
            for node in comp:
                partition[node] = i

    comm_members   = defaultdict(list)
    for node, cid in partition.items():
        comm_members[cid].append(node)
    cluster_size   = {cid: len(m) for cid, m in comm_members.items()}
    cluster_cheat  = {
        cid: (sum(1 for m in members if m in cheat_set) / max(sum(1 for m in members if m in labeled_set), 1))
        for cid, members in comm_members.items()
    }

    user_idxs = np.array([node2idx.get(u, -1) for u in all_users])
    in_graph  = user_idxs >= 0
    safe      = np.where(in_graph, user_idxs, 0)

    c1 = np.where(in_graph, cheat_1hop[safe], 0.0)
    l1 = np.where(in_graph, labeled_1hop[safe], 0.0)
    c2 = np.where(in_graph, A2_cheat[safe], 0.0)
    l2 = np.where(in_graph, A2_labeled[safe], 0.0)
    gc = np.where(in_graph, ghost_1hop[safe], 0.0).astype(int)
    du = np.where(in_graph, degree_arr[safe], 1.0)

    graph_feats = pd.DataFrame({
        'user_hash':              all_users,
        'graph_degree':           [deg_dict.get(u, 0) for u in all_users],
        'neigh_cheat_rate_1hop':  np.where(l1 > 0, c1 / l1, 0.0),
        'cheat_neigh_count_1hop': c1.astype(int),
        'neigh_cheat_rate_2hop':  np.where(l2 > 0, c2 / l2, 0.0),
        'cheat_neigh_count_2hop': c2.astype(int),
        'ghost_neigh_count':      gc,
        'ghost_neigh_ratio':      np.where(du > 0, gc / du, 0.0),
        'louvain_cluster_id':     [partition.get(u, -1) for u in all_users],
        'cluster_size':           [cluster_size.get(partition.get(u, -1), 1) for u in all_users],
        'cluster_cheat_rate':     [cluster_cheat.get(partition.get(u, -1), 0.0) for u in all_users],
    }).set_index('user_hash')

    train_lab_feat   = train_lab_feat.merge(graph_feats, left_on='user_hash', right_index=True, how='left')
    train_unlab_feat = train_unlab_feat.merge(graph_feats, left_on='user_hash', right_index=True, how='left')
    test_feat        = test_feat.merge(graph_feats, left_on='user_hash', right_index=True, how='left')
    print(f'[graph] Done ({time.time()-t0:.1f}s)')
else:
    print('[graph] social_graph.csv not found — skipping graph features')

## 4 — Build Feature Matrices

In [ ]:
meta_cols    = ['user_hash', 'is_cheating', 'high_conf_clean']
feature_cols = [c for c in train_lab_feat.columns if c not in meta_cols]

def clean(df):
    return df[feature_cols].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0)

X_labeled = clean(train_lab_feat)
y_labeled = train_lab_feat['is_cheating'].astype(int)
X_unlab   = clean(train_unlab_feat)
X_test    = clean(test_feat)

print(f'Feature matrix: {X_labeled.shape}  |  Cheating rate: {y_labeled.mean():.4f}')
print(f'Total features: {len(feature_cols)}')

## 5 — Calibration Helpers

In [ ]:
class TemperatureScaler:
    def __init__(self): self.T = 1.0
    def fit(self, y, probs):
        probs = np.clip(probs, 1e-7, 1 - 1e-7)
        lg = logit(probs)
        def nll(T):
            if T <= 0: return 1e9
            s = np.clip(expit(lg / T), 1e-7, 1 - 1e-7)
            return -np.mean(y * np.log(s) + (1 - y) * np.log(1 - s))
        self.T = minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded').x
        return self
    def predict(self, probs):
        return expit(logit(np.clip(probs, 1e-7, 1 - 1e-7)) / self.T)


class PlattScaler:
    def __init__(self): self._lr = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
    def fit(self, y, probs): self._lr.fit(probs.reshape(-1,1), y.astype(int)); return self
    def predict(self, probs): return self._lr.predict_proba(probs.reshape(-1,1))[:, 1]


class IsotonicScaler:
    def __init__(self): self._iso = IsotonicRegression(out_of_bounds='clip')
    def fit(self, y, probs): self._iso.fit(probs, y); return self
    def predict(self, probs): return self._iso.predict(probs)


def select_calibrator(y_cal, probs_cal):
    n_pos = int(y_cal.sum())
    if n_pos < 100:   cal, tag = TemperatureScaler().fit(y_cal, probs_cal), 'temperature'
    elif n_pos < 500: cal, tag = PlattScaler().fit(y_cal, probs_cal),       'platt'
    else:             cal, tag = IsotonicScaler().fit(y_cal, probs_cal),     'isotonic'
    print(f'  [cal] {tag} ({n_pos} positives)')
    return cal


class CalibratedModel:
    def __init__(self, model, calibrator): self.model = model; self.cal = calibrator
    def predict_proba(self, X):
        raw = self.model.predict_proba(X)[:, 1]
        cal = self.cal.predict(raw)
        return np.column_stack([1 - cal, cal])


def train_oof(model, X, y, n_splits=N_SPLITS, cal_frac=0.1):
    skf       = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof_probs = np.zeros(len(X))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_val      = X.iloc[val_idx]
        X_fit, X_cal, y_fit, y_cal = train_test_split(
            X_tr, y_tr, test_size=cal_frac, stratify=y_tr, random_state=RANDOM_STATE,
        )
        m = copy.deepcopy(model)
        m.fit(X_fit.fillna(0), y_fit)
        cal = select_calibrator(np.asarray(y_cal), m.predict_proba(X_cal.fillna(0))[:, 1])
        oof_probs[val_idx] = np.clip(cal.predict(m.predict_proba(X_val.fillna(0))[:, 1]), 0, 1)
        print(f'  fold {fold+1}/{n_splits}  val={len(val_idx)}')

    X_ff, X_fc, y_ff, y_fc = train_test_split(
        X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE,
    )
    final = copy.deepcopy(model)
    final.fit(X_ff.fillna(0), y_ff)
    final_cal = select_calibrator(np.asarray(y_fc), final.predict_proba(X_fc.fillna(0))[:, 1])
    return oof_probs, CalibratedModel(final, final_cal)


print('Calibration & OOF utilities ready')

## 6 — Evaluation Utilities

In [ ]:
def competition_cost(y_true, y_prob, t_pass, t_block):
    y_true, y_prob = np.asarray(y_true, dtype=int), np.asarray(y_prob)
    ap = y_prob < t_pass
    ab = y_prob >= t_block
    rv = ~ap & ~ab
    return float(
        COST_FN        * ((y_true == 1) & ap).sum()
      + COST_FP_BLOCK  * ((y_true == 0) & ab).sum()
      + COST_FP_REVIEW * ((y_true == 0) & rv).sum()
      + COST_TP_REVIEW * ((y_true == 1) & rv).sum()
    )


def find_best_thresholds(y_true, y_prob, step=0.01):
    best_cost, best_t1, best_t2 = float('inf'), None, None
    for t1 in np.arange(0.01, 0.50, step):
        for t2 in np.arange(0.50, 0.99, step):
            if t1 >= t2: continue
            c = competition_cost(y_true, y_prob, t1, t2)
            if c < best_cost:
                best_cost, best_t1, best_t2 = c, t1, t2
    return best_t1, best_t2, best_cost


def evaluate_ensemble(y_true, y_prob, label='ensemble'):
    t_pass, t_block, cost = find_best_thresholds(y_true, y_prob)
    print(f'[{label}] t_pass={t_pass:.3f}  t_block={t_block:.3f}  min_cost=${cost:,.0f}')
    return {'t_pass': t_pass, 't_block': t_block, 'min_cost': cost}


def three_zone_report(y_true, y_prob, t_pass, t_block):
    y_true, y_prob = np.asarray(y_true, dtype=int), np.asarray(y_prob)
    ap = y_prob < t_pass
    ab = y_prob >= t_block
    rv = ~ap & ~ab
    costs = {'auto_pass': {0: 0, 1: COST_FN},
             'review':    {0: COST_FP_REVIEW, 1: COST_TP_REVIEW},
             'auto_block':{0: COST_FP_BLOCK,  1: 0}}
    print(f"\n{'Zone':<15} {'Legit':>8} {'Cheater':>8} {'Total':>8} {'Cost':>10}")
    print('-' * 55)
    total = 0
    for zone_name, mask in [('auto_pass', ap), ('review', rv), ('auto_block', ab)]:
        nl = ((y_true == 0) & mask).sum()
        nc = ((y_true == 1) & mask).sum()
        zc = costs[zone_name][0] * nl + costs[zone_name][1] * nc
        total += zc
        print(f'{zone_name:<15} {nl:>8,} {nc:>8,} {mask.sum():>8,} ${zc:>9,}')
    print('-' * 55)
    print(f"{'TOTAL':<15} {(y_true==0).sum():>8,} {(y_true==1).sum():>8,} {len(y_true):>8,} ${total:>9,}")
    return total


print('Evaluation utilities ready')

## 7 — Model Training

In [ ]:
results = {}

print('=' * 60)
print('1/5  Logistic Regression')
print('=' * 60)
lr_oof, lr_final = train_oof(
    LogisticRegression(C=1.0, class_weight='balanced', solver='lbfgs',
                       max_iter=1000, random_state=RANDOM_STATE),
    X_labeled, y_labeled,
)
results['LogReg'] = evaluate_ensemble(y_labeled.values, lr_oof, 'LogReg')

In [ ]:
print('=' * 60)
print('2/5  Decision Tree')
print('=' * 60)
dt_oof, dt_final = train_oof(
    DecisionTreeClassifier(max_depth=8, min_samples_leaf=20,
                           class_weight='balanced', random_state=RANDOM_STATE),
    X_labeled, y_labeled,
)
results['DecisionTree'] = evaluate_ensemble(y_labeled.values, dt_oof, 'DecisionTree')

In [ ]:
print('=' * 60)
print('3/5  Random Forest')
print('=' * 60)
rf_oof, rf_final = train_oof(
    RandomForestClassifier(n_estimators=500, max_depth=10, min_samples_leaf=10,
                           class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    X_labeled, y_labeled,
)
results['RandomForest'] = evaluate_ensemble(y_labeled.values, rf_oof, 'RandomForest')

In [ ]:
print('=' * 60)
print('4/5  XGBoost')
print('=' * 60)
xgb_oof, xgb_final = train_oof(
    XGBClassifier(
        n_estimators=1000, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        gamma=0.1, scale_pos_weight=COST_RATIO,
        eval_metric='logloss', random_state=RANDOM_STATE,
        n_jobs=-1, tree_method='hist',
    ),
    X_labeled, y_labeled,
)
results['XGBoost'] = evaluate_ensemble(y_labeled.values, xgb_oof, 'XGBoost')

In [ ]:
print('=' * 60)
print('5/5  LightGBM')
print('=' * 60)
lgbm_oof, lgbm_final = train_oof(
    LGBMClassifier(
        n_estimators=1000, learning_rate=0.05, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
        is_unbalance=True, random_state=RANDOM_STATE,
        n_jobs=-1, verbose=-1,
    ),
    X_labeled, y_labeled,
)
results['LightGBM'] = evaluate_ensemble(y_labeled.values, lgbm_oof, 'LightGBM')

## 8 — Ensemble + Pseudo-labeling

In [ ]:
print('=' * 60)
print('Ensemble: XGBoost + LightGBM equal-weight blend')
print('=' * 60)
oof_blend = (xgb_oof + lgbm_oof) / 2
results['Ensemble'] = evaluate_ensemble(y_labeled.values, oof_blend, 'Ensemble')

In [ ]:
# Semi-supervised: pseudo-label high-confidence clean unlabeled users
if 'high_conf_clean' in train_unlab_feat.columns:
    high_conf_mask = train_unlab_feat['high_conf_clean'].fillna(0).astype(bool)
    X_candidates   = X_unlab[high_conf_mask]
    print(f'[pseudo] High-conf candidates: {len(X_candidates):,}')

    if len(X_candidates) > 0:
        probs       = xgb_final.predict_proba(X_candidates.fillna(0))[:, 1]
        pseudo_mask = probs < 0.05
        X_pseudo    = X_candidates[pseudo_mask]
        y_pseudo    = pd.Series(0, index=X_pseudo.index, dtype=int)
        print(f'[pseudo] Pseudo-negatives added: {pseudo_mask.sum():,}')

        if len(X_pseudo) > 0:
            X_expanded = pd.concat([X_labeled, X_pseudo]).reset_index(drop=True)
            y_expanded = pd.concat([y_labeled, y_pseudo]).reset_index(drop=True)
            n_orig     = len(y_labeled)

            print('=' * 60)
            print('XGBoost retrained with pseudo-negatives')
            print('=' * 60)
            xgb2_oof, xgb2_final = train_oof(
                XGBClassifier(
                    n_estimators=1000, learning_rate=0.05, max_depth=6,
                    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                    gamma=0.1, scale_pos_weight=COST_RATIO,
                    eval_metric='logloss', random_state=RANDOM_STATE,
                    n_jobs=-1, tree_method='hist',
                ),
                X_expanded, y_expanded,
            )
            oof_blend = (xgb2_oof[:n_orig] + lgbm_oof) / 2
            xgb_final = xgb2_final
            results['Ensemble+Pseudo'] = evaluate_ensemble(
                y_labeled.values, oof_blend, 'Ensemble+Pseudo'
            )
else:
    print('[pseudo] high_conf_clean column not found — skipping')

## 9 — Model Comparison & Best Selection

In [ ]:
print(f"{'Model':<22} {'t_pass':>8} {'t_block':>8} {'Cost':>12}")
print('-' * 55)
for name, res in results.items():
    print(f"{name:<22} {res['t_pass']:>8.3f} {res['t_block']:>8.3f} ${res['min_cost']:>10,.0f}")

best_name = min(results, key=lambda k: results[k]['min_cost'])
best      = results[best_name]
print(f'\nBest: {best_name}  |  Cost: ${best["min_cost"]:,.0f}')

ensemble_names = {'Ensemble', 'Ensemble+Pseudo'}
best_oof = oof_blend if best_name in ensemble_names else (
    xgb_oof if best_name == 'XGBoost' else (
    lgbm_oof if best_name == 'LightGBM' else oof_blend))

three_zone_report(y_labeled.values, best_oof, best['t_pass'], best['t_block'])

y_pred05 = (best_oof >= 0.5).astype(int)
print(f"\nAUC-ROC:   {roc_auc_score(y_labeled, best_oof):.4f}")
print(f"F1:        {f1_score(y_labeled, y_pred05, zero_division=0):.4f}")
print(f"Precision: {precision_score(y_labeled, y_pred05, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_labeled, y_pred05, zero_division=0):.4f}")

## 10 — Generate Submission

In [ ]:
print('[pipeline] Generating test predictions ...')
xgb_test    = xgb_final.predict_proba(X_test.fillna(0))[:, 1]
lgbm_test   = lgbm_final.predict_proba(X_test.fillna(0))[:, 1]
final_preds = (xgb_test + lgbm_test) / 2

submission = pd.DataFrame({
    'user_hash':  test['user_hash'].values,
    'prediction': final_preds,
})

submission.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Saved: /kaggle/working/submission.csv  ({len(submission)} rows)')
submission.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(best_oof, bins=60, edgecolor='k', alpha=0.75)
axes[0].axvline(best['t_pass'],  color='green', ls='--', lw=1.5, label=f"t_pass={best['t_pass']:.2f}")
axes[0].axvline(best['t_block'], color='red',   ls='--', lw=1.5, label=f"t_block={best['t_block']:.2f}")
axes[0].set_title('OOF Probability Distribution')
axes[0].set_xlabel('Predicted probability'); axes[0].legend()

axes[1].hist(final_preds, bins=60, edgecolor='k', alpha=0.75, color='steelblue')
axes[1].axvline(best['t_pass'],  color='green', ls='--', lw=1.5, label=f"t_pass={best['t_pass']:.2f}")
axes[1].axvline(best['t_block'], color='red',   ls='--', lw=1.5, label=f"t_block={best['t_block']:.2f}")
axes[1].set_title('Test Prediction Distribution')
axes[1].set_xlabel('Predicted probability'); axes[1].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/prediction_distributions.png', dpi=150)
plt.show()
print('Done.')